In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def find_page(isbn):
    search_url = r"https://www.aladin.co.kr/search/wsearchresult.aspx?SearchTarget=All&SearchWord={}"
    headers = {"User-Agent": "Mozilla/5.0"} #난 봇이 아니다.
    try: 
        former_page = requests.get(search_url.format(isbn), headers=headers, timeout=10)
        former_page.raise_for_status() #http 페이지 오류일시 except로 넘어가게 하는 것
    except requests.RequestException:
        return None, "search_reques_fail" #페이지 오류시 함수 전체 종료
    
    fpage_soup = BeautifulSoup(former_page.text, "html.parser") #HTML을 트리구조로 만들어 find, find_all, select 사용가능
    sp_link = fpage_soup.find("a", attrs={"class":"bo3"})
    if sp_link is None:
        return None, "no_bo3_link"
    href = sp_link.get("href")
    if not href:
        return None, "no_href"
    fpage_soup()
    
    try:
        detail_page = requests.get(href, headers=headers, timeout=10)
        detail_page.raise_for_status()
    except requests.RequestException as e:
        print(f"[FAIL] isbn={isbn} error={type(e).__name__}: {e}")
        return None, "detail_request_fail"
    
    detail_page_soup = BeautifulSoup(detail_page.text, 'html.parser')
    div_box = detail_page_soup.find('div', attrs={'class':'Ere_prod_mconts_R'})
    if not div_box:
        return None, "no_div_box"
    
    ul_list = div_box.find_all('ul')
    li_list = []
    for ul in ul_list:
        li_list.extend(ul.find_all('li'))
    
    for li in li_list:
        li_text = li.get_text(" ", strip=True)
        if "쪽" in li_text:
            idx = li_text.find('쪽')
            page_digits = ""
            i = idx-1
            while i>=0 and li_text[i].isdigit():
                page_digits = li_text[i] + page_digits
                i -= 1
            if page_digits:
                return int(page_digits)


#데이터프레임 생성            
books_df = pd.read_json(r"C:\data\2030s_bestbook_df.json")
books_page_df = books_df.apply(lambda row: find_page(row['isbn13']), axis=1)
books_page_df.name = 'page_count'
books_with_page_df = pd.merge(books_df, books_page_df, left_index=True, right_index=True)
books_with_page_df.to_csv("2030s_bestbook_df_page", index=False, encoding='utf-8')